In [1]:
# Install dependencies if needed
# !pip install requests pandas python-dotenv

import os
import time
import requests
import pandas as pd
from datetime import date, timedelta

In [2]:
#API_KEY = os.getenv("MASSIVE_API_KEY", "")

# Optional temporary method. Uncomment and paste your key if you have not set an environment variable.
API_KEY = "xsJlMe_F2PsGLr7EJsOHOBP0VpMb7Rsy"

if not API_KEY:
    raise ValueError("No API key found. Set MASSIVE_API_KEY or paste it into API_KEY above.")

BASE_URL = "https://api.massive.com"
print("API key loaded. Length:", len(API_KEY))

API key loaded. Length: 32


In [3]:
def massive_get(path, params=None, sleep_seconds=0.0):
    # Simple Massive API GET helper.
    if params is None:
        params = {}
    params = dict(params)
    params["apiKey"] = API_KEY
    url = BASE_URL + path
    response = requests.get(url, params=params, timeout=30)
    print("GET", response.url.replace(API_KEY, "***API_KEY***"))
    print("Status:", response.status_code)
    
    if sleep_seconds:
        time.sleep(sleep_seconds)
    
    try:
        data = response.json()
    except Exception:
        print(response.text[:500])
        response.raise_for_status()
        return None
    
    if response.status_code >= 400:
        print(data)
        response.raise_for_status()
    
    return data


def results_to_df(data):
    # Convert common Massive response format into a DataFrame.
    if not data:
        return pd.DataFrame()
    results = data.get("results", [])
    if isinstance(results, dict):
        results = [results]
    return pd.DataFrame(results)

In [4]:
index_tickers_to_test = ["I:SPX", "I:VIX"]

for ticker in index_tickers_to_test:
    print("\n--- Checking", ticker, "---")
    data = massive_get("/v3/reference/tickers", params={"ticker": ticker, "market": "indices"}, sleep_seconds=1)
    df = results_to_df(data)
    display(df.head())


--- Checking I:SPX ---
GET https://api.massive.com/v3/reference/tickers?ticker=I%3ASPX&market=indices&apiKey=***API_KEY***
Status: 200


,ticker,name,market,locale,active,source_feed
0,I:SPX,Standard & Poor's 500,indices,us,True,CboeGlobalIndicesMain



--- Checking I:VIX ---
GET https://api.massive.com/v3/reference/tickers?ticker=I%3AVIX&market=indices&apiKey=***API_KEY***
Status: 200


,ticker,name,market,locale,active,source_feed
0,I:VIX,Cboe Volatility Index,indices,us,True,CboeGlobalIndicesMain


In [5]:
# Change these dates to a recent weekday if the request returns no data.
from_date = "2026-06-01"
to_date = "2026-06-01"

for ticker in ["I:SPX", "I:VIX"]:
    print("\n--- Minute aggregates for", ticker, "---")
    path = f"/v2/aggs/ticker/{ticker}/range/1/minute/{from_date}/{to_date}"
    data = massive_get(path, params={"adjusted": "true", "sort": "asc", "limit": 5000}, sleep_seconds=1)
    df = results_to_df(data)
    print("Rows:", len(df))
    display(df.head())


--- Minute aggregates for I:SPX ---
GET https://api.massive.com/v2/aggs/ticker/I:SPX/range/1/minute/2026-06-01/2026-06-01?adjusted=true&sort=asc&limit=5000&apiKey=***API_KEY***
Status: 403
{'status': 'NOT_AUTHORIZED', 'request_id': 'd4769e0d2f8044baad5c159dd53f43e6', 'message': 'You are not entitled to this data. Please upgrade your plan at https://massive.com/pricing'}


HTTPError: 403 Client Error: Forbidden for url: https://api.massive.com/v2/aggs/ticker/I:SPX/range/1/minute/2026-06-01/2026-06-01?adjusted=true&sort=asc&limit=5000&apiKey=xsJlMe_F2PsGLr7EJsOHOBP0VpMb7Rsy

In [6]:
print("--- Checking SPY stock/ETF ticker ---")
data = massive_get("/v3/reference/tickers", params={"ticker": "SPY", "market": "stocks"}, sleep_seconds=1)
df = results_to_df(data)
display(df.head())

--- Checking SPY stock/ETF ticker ---
GET https://api.massive.com/v3/reference/tickers?ticker=SPY&market=stocks&apiKey=***API_KEY***
Status: 200


,ticker,name,market,locale,primary_exchange,type,active,currency_name,cik,composite_figi,share_class_figi,last_updated_utc
0,SPY,State Street SPDR S&P 500 ETF Trust,stocks,us,ARCX,ETF,True,usd,0000884394,BBG000BDTBL9,BBG001S72SM3,2026-07-06T06:09:14.586337131Z


In [7]:
print("--- SPY minute aggregates ---")
path = f"/v2/aggs/ticker/SPY/range/1/minute/{from_date}/{to_date}"
data = massive_get(path, params={"adjusted": "true", "sort": "asc", "limit": 5000}, sleep_seconds=1)
spy_min = results_to_df(data)
print("Rows:", len(spy_min))
display(spy_min.head())

--- SPY minute aggregates ---
GET https://api.massive.com/v2/aggs/ticker/SPY/range/1/minute/2026-06-01/2026-06-01?adjusted=true&sort=asc&limit=5000&apiKey=***API_KEY***
Status: 200
Rows: 906


,v,vw,o,c,h,l,t,n
0,8785.917677,758.4792,758.02,758.18,759.00,757.79,1780300800000,895
1,10945.000000,758.6007,758.06,758.26,759.00,758.06,1780300860000,157
2,815.091803,758.3001,758.23,758.20,758.23,758.20,1780300920000,26
3,43142.099783,758.5285,758.38,758.23,759.08,757.83,1780300980000,1406
4,755.020876,758.1940,758.20,758.06,758.23,758.06,1780301040000,33


In [8]:
def search_option_contracts(underlying_ticker="I:SPX", expiration_date=None, contract_type=None, limit=10):
    params = {
        "underlying_ticker": underlying_ticker,
        "limit": limit,
        "sort": "expiration_date",
        "order": "asc"
    }
    if expiration_date:
        params["expiration_date"] = expiration_date
    if contract_type:
        params["contract_type"] = contract_type
    data = massive_get("/v3/reference/options/contracts", params=params, sleep_seconds=1)
    return results_to_df(data)

print("--- Trying underlying_ticker='I:SPX' ---")
spx_opts_i = search_option_contracts("I:SPX", limit=10)
print("Rows:", len(spx_opts_i))
display(spx_opts_i.head())

if spx_opts_i.empty:
    print("\nNo contracts returned for I:SPX. Trying underlying_ticker='SPX'...")
    spx_opts_plain = search_option_contracts("SPX", limit=10)
    print("Rows:", len(spx_opts_plain))
    display(spx_opts_plain.head())
else:
    spx_opts_plain = pd.DataFrame()

--- Trying underlying_ticker='I:SPX' ---
GET https://api.massive.com/v3/reference/options/contracts?underlying_ticker=I%3ASPX&limit=10&sort=expiration_date&order=asc&apiKey=***API_KEY***
Status: 200
Rows: 0


""



No contracts returned for I:SPX. Trying underlying_ticker='SPX'...
GET https://api.massive.com/v3/reference/options/contracts?underlying_ticker=SPX&limit=10&sort=expiration_date&order=asc&apiKey=***API_KEY***
Status: 200
Rows: 10


,cfi,contract_type,exercise_style,expiration_date,primary_exchange,shares_per_contract,strike_price,ticker,underlying_ticker
0,OCEICS,call,european,2026-07-06,XCBO,100,3000,O:SPXW260706C03000000,SPX
1,OCEICS,call,european,2026-07-06,XCBO,100,3200,O:SPXW260706C03200000,SPX
2,OCEICS,call,european,2026-07-06,XCBO,100,3400,O:SPXW260706C03400000,SPX
3,OCEICS,call,european,2026-07-06,XCBO,100,3600,O:SPXW260706C03600000,SPX
4,OCEICS,call,european,2026-07-06,XCBO,100,3800,O:SPXW260706C03800000,SPX


In [9]:
print("--- Trying SPY options ---")
spy_opts = search_option_contracts("SPY", limit=10)
print("Rows:", len(spy_opts))
display(spy_opts.head())

--- Trying SPY options ---
GET https://api.massive.com/v3/reference/options/contracts?underlying_ticker=SPY&limit=10&sort=expiration_date&order=asc&apiKey=***API_KEY***
Status: 200
Rows: 10


,cfi,contract_type,exercise_style,expiration_date,primary_exchange,shares_per_contract,strike_price,ticker,underlying_ticker
0,OCASPS,call,american,2026-07-06,BATO,100,500,O:SPY260706C00500000,SPY
1,OCASPS,call,american,2026-07-06,BATO,100,505,O:SPY260706C00505000,SPY
2,OCASPS,call,american,2026-07-06,BATO,100,510,O:SPY260706C00510000,SPY
3,OCASPS,call,american,2026-07-06,BATO,100,515,O:SPY260706C00515000,SPY
4,OCASPS,call,american,2026-07-06,BATO,100,520,O:SPY260706C00520000,SPY


In [10]:
contract_df = None
source_name = None

if 'spx_opts_i' in globals() and not spx_opts_i.empty:
    contract_df = spx_opts_i
    source_name = "SPX via I:SPX"
elif 'spx_opts_plain' in globals() and not spx_opts_plain.empty:
    contract_df = spx_opts_plain
    source_name = "SPX via SPX"
elif 'spy_opts' in globals() and not spy_opts.empty:
    contract_df = spy_opts
    source_name = "SPY fallback"

if contract_df is None or contract_df.empty:
    raise ValueError("No option contracts found. Check plan access, ticker format, or API key.")

sample_contract = contract_df.iloc[0].to_dict()
print("Using", source_name)
print("Sample contract:")
for k in ["ticker", "underlying_ticker", "contract_type", "expiration_date", "strike_price", "exercise_style"]:
    if k in sample_contract:
        print(f"{k}: {sample_contract[k]}")

option_ticker = sample_contract["ticker"]
option_ticker

Using SPX via SPX
Sample contract:
ticker: O:SPXW260706C03000000
underlying_ticker: SPX
contract_type: call
expiration_date: 2026-07-06
strike_price: 3000
exercise_style: european


'O:SPXW260706C03000000'

In [11]:
# Use the option expiration date if available, otherwise use the same test date.
agg_date = sample_contract.get("expiration_date", from_date)
print("Requesting aggregates for", option_ticker, "on", agg_date)

path = f"/v2/aggs/ticker/{option_ticker}/range/1/minute/{agg_date}/{agg_date}"
data = massive_get(path, params={"adjusted": "true", "sort": "asc", "limit": 5000}, sleep_seconds=1)
opt_min = results_to_df(data)
print("Rows:", len(opt_min))
display(opt_min.head())

Requesting aggregates for O:SPXW260706C03000000 on 2026-07-06
GET https://api.massive.com/v2/aggs/ticker/O:SPXW260706C03000000/range/1/minute/2026-07-06/2026-07-06?adjusted=true&sort=asc&limit=5000&apiKey=***API_KEY***
Status: 403
{'status': 'NOT_AUTHORIZED', 'request_id': '5812b0f8fae65a24ef2fa55f69c251b0', 'message': "Your plan doesn't include this data timeframe. Please upgrade your plan at https://polygon.io/pricing"}


HTTPError: 403 Client Error: Forbidden for url: https://api.massive.com/v2/aggs/ticker/O:SPXW260706C03000000/range/1/minute/2026-07-06/2026-07-06?adjusted=true&sort=asc&limit=5000&apiKey=xsJlMe_F2PsGLr7EJsOHOBP0VpMb7Rsy


Based on the plan features visible on Massive pricing:

| Data item | Free/basic check in this notebook? | Meaning for dissertation |
|---|---:|---|
| SPY minute aggregates | Yes, via Stocks Basic | Good for SPY volume/VWAP proxy features |
| SPX/VIX minute aggregates | Maybe, via Indices Basic | Depends on limited index ticker access and history |
| Options contract reference data | Yes, via Options Basic | Lets you find SPX/SPY option contracts |
| Option minute aggregates | Yes, via Options Basic | Gives rough option OHLCV movement |
| Greeks / IV | No on Options Basic | Needed for gamma/theta/IV analysis and GEX |
| Daily open interest | No on Options Basic | Needed for GEX estimation |
| Trades | No on Options Basic/Starter | Useful for actual transaction prints |
| Quotes / bid-ask | No on Options Basic/Starter | Needed for spread, slippage and realistic 0DTE execution |
| GEX | Not directly | Must be calculated or sourced elsewhere |
